# ARIES Thermal Assets Alfalfa Example

Tests the `Boiler` and `BuriedPipePair` FMUs from the [FMU-compilation-workbench](https://github.com/NatLabRockies/LDRD-low-temp-thermal-ARIES) project by uploading both to Alfalfa as independent sites and stepping them through time as a simple, manually-coupled district heating loop:

```
Boiler outlet -> BuriedPipePair supply inlet -> (load, not modeled) -> BuriedPipePair return inlet (fixed) -> BuriedPipePair return outlet -> Boiler inlet
```

## Setup
1. [Alfalfa stack deployed](https://github.com/NatLabRockies/alfalfa/wiki/Deployment) locally (or have the url for the Alfalfa host you are using) with **at least two workers**, since this notebook runs both FMUs simultaneously.
2. The `.fmu` files in `./aries_thermal_assets/` are compiled outputs of the FMU-compilation-workbench:
   - `Boiler_EC.fmu` — `Buildings.Fluid.Boilers.BoilerPolynomial` (natural gas boiler, polynomial efficiency curve)
   - `BuriedPipePair_FMU.fmu` — `Buildings.Fluid.FixedResistances.BuriedPipes.GroundCoupling` (coupled supply/return pipes buried in a shared trench)

**Troubleshooting:** these FMUs were compiled with the workbench's default `--solver cvode` (SUNDIALS BDF implicit solver). If your Alfalfa worker image is missing the SUNDIALS shared libraries, `ac.start()` will fail with `InvalidBinaryException: ... libsundials_nvecserial.so.5: cannot open shared object file`. Either add `libsundials-nvecserial6`/`libsundials-cvode6` (or copy the `.so.5` files from the `fmu-workbench` image's `/usr/lib/x86_64-linux-gnu/omc/`) into the worker container, or recompile the FMUs with `--solver euler`/`--solver implicit_euler` (see the FMU-compilation-workbench `Makefile`/`README.md`).

## Notes on model I/O
Point names below come from each FMU's `modelDescription.xml` (Alfalfa exposes the raw FMI variable names as points).

**Boiler** (`Boiler_EC.fmu`)
| Point | Causality | Unit | Description |
| --- | --- | --- | --- |
| `inlet.forward.T` | Input | K | Inlet (return) water temperature |
| `inlet.m_flow` | Input | kg/s | Water mass flow rate |
| `y` | Input | 0-1 | Burner modulation / control signal |
| `outlet.forward.T` | Output | K | Outlet (supply) water temperature |
| `outlet.m_flow` | Output | kg/s | Outlet mass flow rate |

**BuriedPipePair** (`BuriedPipePair_FMU.fmu`)
| Point | Causality | Unit | Description |
| --- | --- | --- | --- |
| `T_in_sup` | Input | K | Supply pipe inlet temperature |
| `m_flow_sup` | Input | kg/s | Supply pipe mass flow rate |
| `T_in_ret` | Input | K | Return pipe inlet temperature |
| `m_flow_ret` | Input | kg/s | Return pipe mass flow rate |
| `T_out_sup` | Output | K | Supply pipe outlet temperature |
| `T_out_ret` | Output | K | Return pipe outlet temperature |
| `Q_sup` | Output | W | Supply pipe heat loss to ground |
| `Q_ret` | Output | W | Return pipe heat loss/gain to ground |

Nominal values used below (0.5 kg/s flow, 343.15 K / 70&deg;C supply, 313.15 K / 40&deg;C return) match the `m1_flow_nominal`/`m2_flow_nominal` defaults and documented start values in the FMU-compilation-workbench datasheets (`catalog/models/Boiler/README.md` and `catalog/models/BuriedPipePair/README.md`).

**Timestep note:** Alfalfa's Modelica/FMU worker currently advances FMUs on a fixed 1-minute step regardless of the FMU's recommended real-time timestep (2.0 s for the boiler, 0.1 s for the buried pipe pair per their datasheets), so this is a supervisory-scale test rather than a full-bandwidth HIL test.

In [ ]:
import datetime
from pathlib import Path
from pprint import pprint

import pandas as pd
import matplotlib.pyplot as plt

from alfalfa_client.alfalfa_client import AlfalfaClient


### Create new Alfalfa client object
If you are not hosting the Alfalfa server yourself replace the `host` with the one of your server without a trailing slash (this is a known bug and will eventually be fixed)

In [ ]:
ac = AlfalfaClient(host='http://localhost')


### Define paths to models to be uploaded
Both models are single `.fmu` files living alongside this notebook in `./aries_thermal_assets/`.

In [ ]:
model_paths = [
    str(Path('./aries_thermal_assets/Boiler_EC.fmu')),
    str(Path('./aries_thermal_assets/BuriedPipePair_FMU.fmu')),
]
model_paths

### Upload sites to Alfalfa
The `ac.submit` function returns a `site_id` (here, a list of two) used to interact with each site over the API. A simulation is a site.

Sites can be viewed at http://localhost/sites

In [ ]:
site_ids = ac.submit(model_paths)
boiler_id, pipe_id = site_ids
print(f"boiler_id: {boiler_id}")
print(f"pipe_id:   {pipe_id}")

### Define parameters to run the simulations
`external_clock=True` so this notebook advances the models itself via `ac.advance()` rather than Alfalfa advancing them on a wall-clock timescale.

In [ ]:
start_dt = datetime.datetime(2024, 1, 1, 0, 0, 0)
end_dt = start_dt + datetime.timedelta(hours=1)

params = {
    "external_clock": True,
    "start_datetime": start_dt,
    "end_datetime": end_dt,
}

## Start simulations
Note: one sim runs / worker, so if you have not scaled your local deployment to workers >= 2, the second site won't have a chance to start and this code block will not complete.

In [ ]:
print(f"Starting sites: {site_ids}")
ac.start(site_ids, **params)

### Get each model's input points

In [ ]:
print(f"{boiler_id} (boiler) inputs:")
pprint(ac.get_inputs(boiler_id))
print(f"{pipe_id} (buried pipe pair) inputs:")
pprint(ac.get_inputs(pipe_id))

### Get each model's output points
Before advancing, these reflect the FMUs' start values (see the tables above).

In [ ]:
print(f"{boiler_id} (boiler) outputs:")
pprint(ac.get_outputs(boiler_id))
print(f"{pipe_id} (buried pipe pair) outputs:")
pprint(ac.get_outputs(pipe_id))

## Step through time with a coupled boiler + buried pipe loop

Each timestep:
1. Set the boiler's inlet temperature to the buried return pipe's outlet temperature from the *previous* step (closing the loop), hold flow and firing rate (`y`) fixed, then advance the boiler.
2. Feed the boiler's fresh outlet temperature into the buried pipe pair's supply inlet. The pipe pair's return inlet is held at a fixed 313.15 K (40&deg;C), representing a constant downstream load return condition (no building model in this test). Advance the pipe pair.
3. Record both models' outputs.

This is a simple single-pass (explicit) coupling — good enough for this supervisory-scale test, though a stiffer/faster-dynamics coupling would benefit from iterating each timestep until the interface values converge.

In [ ]:
# Fixed/nominal conditions, matching the FMU-compilation-workbench datasheets
boiler_m_flow = 0.5     # kg/s, m1_flow_nominal
boiler_u = 0.75         # burner modulation signal (0-1)
pipe_m_flow = 0.5       # kg/s, m1_flow_nominal / m2_flow_nominal
pipe_T_in_ret = 313.15  # K (40 degC) - fixed downstream/load return condition

# Initial guess for the boiler's inlet temperature, matching the buried pipe
# pair's documented T_in_ret start value so the loop starts near steady state
boiler_T_in = 313.15

timesteps = 30
history = []

for _ in range(timesteps):
    ac.set_inputs(boiler_id, {
        "inlet.forward.T": boiler_T_in,
        "inlet.m_flow": boiler_m_flow,
        "y": boiler_u,
    })
    ac.advance(boiler_id)
    boiler_out = ac.get_outputs(boiler_id)
    boiler_T_out = boiler_out["outlet.forward.T"]

    ac.set_inputs(pipe_id, {
        "T_in_sup": boiler_T_out,
        "m_flow_sup": pipe_m_flow,
        "T_in_ret": pipe_T_in_ret,
        "m_flow_ret": pipe_m_flow,
    })
    ac.advance(pipe_id)
    pipe_out = ac.get_outputs(pipe_id)

    # Close the loop for the next timestep
    boiler_T_in = pipe_out["T_out_ret"]

    sim_time = ac.get_sim_time(boiler_id)
    history.append({
        "time": sim_time,
        "boiler_T_out_C": boiler_T_out - 273.15,
        "pipe_T_out_sup_C": pipe_out["T_out_sup"] - 273.15,
        "pipe_T_out_ret_C": pipe_out["T_out_ret"] - 273.15,
        "Q_sup_W": pipe_out["Q_sup"],
        "Q_ret_W": pipe_out["Q_ret"],
    })
    print(f"t={sim_time}  boiler_out={boiler_T_out - 273.15:6.2f}C  "
          f"pipe_sup_out={pipe_out['T_out_sup'] - 273.15:6.2f}C  "
          f"pipe_ret_out={pipe_out['T_out_ret'] - 273.15:6.2f}C")

### Collect results

In [ ]:
df = pd.DataFrame(history).set_index("time")
df

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(df.index, df["boiler_T_out_C"], label="Boiler outlet")
axes[0].plot(df.index, df["pipe_T_out_sup_C"], label="Buried pipe supply outlet")
axes[0].plot(df.index, df["pipe_T_out_ret_C"], label="Buried pipe return outlet")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df.index, df["Q_sup_W"], label="Q_sup (supply pipe loss)")
axes[1].plot(df.index, df["Q_ret_W"], label="Q_ret (return pipe loss)")
axes[1].set_ylabel("Heat loss to ground (W)")
axes[1].set_xlabel("Simulation time")
axes[1].legend()
axes[1].grid(True)

fig.autofmt_xdate()
fig.tight_layout()

### Stop the simulations

In [ ]:
ac.stop(site_ids)